# M1 — Context-Reliability and Risk-Controlled Intrusion Detection

**TRAIN-only CPU experiment; does not read UNSW-NB15 official test.** Registered methodology: [`docs/M1_CONTEXT_RISK_METHOD_PROTOCOL.md`](https://github.com/itsCodeBakery/rare-attack-spatiotemporal-ids/blob/main/docs/M1_CONTEXT_RISK_METHOD_PROTOCOL.md). The 42-feature target expert, 444-feature contextual expert, OOF utility gate and five alpha constraints are predeclared. CPU fit, first seed **42 only**. Future seeds can be run in independent, logged Kaggle sessions after review.

**Instructions:** Add the existing Kaggle dataset `dsshayan/unsw-nb15` to the notebook. Internet must be **ON** for cloning the public research repository. Set `pushRare` in Kaggle Secrets **only** if you want an unpublished draft GitHub Release backup. Archive includes private original development predictions and must never be published.

**Important:** This new split is internal development, not an independent official-test or verified chronological experiment. Do not use the 2026 SCV-IDS paper's official scores as the denominator for M1 experimental improvements.

In [ ]:
import os, sys, subprocess, json
from pathlib import Path

REPO = Path('/kaggle/working/rare-attack-spatiotemporal-ids')
TRAIN = Path('/kaggle/input/datasets/dsshayan/unsw-nb15/UNSW_NB15_training-set.csv')
OUTPUT = Path('/kaggle/working/M1-CONTEXT-RISK-001')
BACKUP = Path('/kaggle/working/m1_private_backups')
SEED = 42

if not TRAIN.is_file():
    raise FileNotFoundError('Attach the dataset dsshayan/unsw-nb15; do NOT substitute test.csv: '+str(TRAIN))
assert TRAIN.name == 'UNSW_NB15_training-set.csv'
if REPO.exists():
    raise RuntimeError('Existing repo checkout: start a fresh Kaggle runtime or verify it manually; do not silently update code.')
subprocess.run(['git','clone','--depth','1','https://github.com/itsCodeBakery/rare-attack-spatiotemporal-ids.git',str(REPO)],check=True)
revision=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
print('FROZEN GIT COMMIT:',revision)
print('TRAIN INPUT ONLY:',TRAIN)
print('GPU is not required; use Kaggle CPU.')

# Use existing Kaggle libraries if present; no silent package upgrades.
for module in ('numpy','pandas','lightgbm','scipy','sklearn','pytest','nbformat'):
    __import__(module)

import hashlib
h=hashlib.sha256()
with TRAIN.open('rb') as f:
    for buf in iter(lambda:f.read(4*1024*1024),b''):h.update(buf)
locked=json.loads((REPO/'configs/m1_context_risk_001.json').read_text())
if h.hexdigest()!=locked['training_csv_sha256']:
    raise ValueError('Training CSV SHA-256 does not match preregistered source')
if locked['official_unsw_test']['access']!='FORBIDDEN':raise ValueError('Unsafe experiment config')
print('TRAINING SOURCE HASH CHECK PASSED. OFFICIAL TEST NOT OPENED.')

In [ ]:
environment=os.environ.copy()
environment['PYTHONPATH']=str(REPO/'src')+os.pathsep+str(REPO)
environment['OMP_NUM_THREADS']='4'
environment['OPENBLAS_NUM_THREADS']='1'
environment['MKL_NUM_THREADS']='1'
print('Running M1 synthetic safety tests...')
subprocess.run([sys.executable,'-m','pytest','-q',str(REPO/'tests/test_m1_synthetic.py'),
                 str(REPO/'tests/test_m1_security.py')],env=environment,cwd=str(REPO),check=True)
print('SYNTHETIC TESTS PASSED — BEGIN FIRST REAL TRAINING-ONLY M1 FIT')

In [ ]:
OUTPUT.mkdir(parents=True,exist_ok=True)
command=[sys.executable,'-m','scripts.run_m1_context_risk',
         '--repo',str(REPO), '--config',str(REPO/'configs/m1_context_risk_001.json'),
         '--train-csv',str(TRAIN), '--output',str(OUTPUT),'--seed',str(SEED)]
print('Running:', ' '.join(command), flush=True)
subprocess.run(command,cwd=str(REPO),env=environment,check=True)
print('M1 development experiment ended. Result audit is not yet complete.')

In [ ]:
# Local checksum-verified private backup even if no GitHub secret is configured.
seed_dir=OUTPUT/f'seed_{SEED}'
command=[sys.executable,str(REPO/'scripts/archive_m1_private.py'),
         '--seed-dir',str(seed_dir),'--archive-dir',str(BACKUP)]
secret=None
try:
    from kaggle_secrets import UserSecretsClient
    secret=UserSecretsClient().get_secret('pushRare')
except Exception:
    print('No pushRare secret available: private backup remains in Kaggle working files.')
if secret:
    environment['GITHUB_TOKEN']=secret
    command.append('--draft-upload')
else:
    print('To preserve archive outside Kaggle, use Save Version or download the private tar.gz yourself.')
subprocess.run(command,cwd=str(REPO),env=environment,check=True)
print('PRIVATE ARCHIVE:',list(BACKUP.glob('*.tar.gz')))
report=json.loads((seed_dir/'final_public_summary.json').read_text())
print('TRAIN-DERIVED DEVELOPMENT METRICS ONLY')
for key,val in report['complete_controls'].items():
    print(f"{key:18s} acc={val['accuracy']:.5f} macroF1={val['macro_f1']:.5f} benignFPR={val['benign_fpr']:.5f}")
print('Selected alpha:',report['selected_alpha'],'calibration:',report['gate_calibration_status'])
print('Next: independent prediction audit and aggregate-only GitHub sync. Do NOT publish draft archives.')